# Your First Mu2e Analysis

This notebook provides a hands-on introduction to analyzing Mu2e EventNtuple data using the Mu2e Python stack.

**Learning objectives:**
- Load and inspect EventNtuple data
- Apply physics selection cuts
- Calculate derived quantities
- Create publication-quality plots
- Track cut flow and efficiencies

**Prerequisites:**
- Mu2e Python environment activated (`conda activate ana_v2.4.0` or `pyenv ana`)
- Access to EventNtuple ROOT files

**Estimated time:** 30-45 minutes

## Setup

First, let's import all the packages we'll need.

In [ ]:
# Standard scientific Python packages
import awkward as ak
import numpy as np
import matplotlib.pyplot as plt

# pyutils modules
from pyutils.pyprocess import Processor
from pyutils.pyprint import Print
from pyutils.pyselect import Select
from pyutils.pycut import CutManager
from pyutils.pyplot import Plot
from pyutils.pyvector import Vector

print("✅ All imports successful!")

## Configuration

Set your input file path and which branches to load.

**Note:** Replace `INPUT_FILE` with the path to your EventNtuple file.

In [ ]:
# ⚠️ EDIT THIS: Set your input file path
INPUT_FILE = "/path/to/your/eventntuple.root"

# Which branches to load (only load what you need!)
BRANCHES = [
    "trk",       # Track information
    "trksegs",   # Track segments
    "trkqual"    # Track quality MVA
]

# Configuration for cuts (we'll use these later)
QUALITY_CUT = 0.8
MOMENTUM_MIN = 100.0  # MeV/c
MOMENTUM_MAX = 110.0  # MeV/c

## Step 1: Load the Data

We'll use the `Processor` class to load our EventNtuple file.

In [ ]:
# Create a processor
proc = Processor(verbosity=1)

# Load the data
print(f"Loading data from: {INPUT_FILE}")
print(f"Branches: {BRANCHES}")
print()

data = proc.process_data(
    file_name=INPUT_FILE,
    branches=BRANCHES
)

print(f"\n✅ Loaded {len(data)} events")

**What just happened?**
- `Processor` opened the ROOT file
- Imported the specified branches into awkward arrays
- Returned a structured array containing all the data

**Troubleshooting:**
- If you get "file not found", check your `INPUT_FILE` path
- If you get "branch not found", check the branch names in your file

## Step 2: Inspect the Data Structure

Before doing any analysis, let's look at what we have.

In [ ]:
# Check the overall structure
print("Data type:")
print(data.type)
print()

# Basic statistics
print("Basic statistics:")
print(f"Total events: {len(data)}")
print(f"Events with at least one track: {ak.sum(ak.num(data.trk) > 0)}")
print(f"Total tracks: {ak.sum(ak.num(data.trk))}")
print(f"Average tracks per event: {ak.sum(ak.num(data.trk)) / len(data):.2f}")

In [ ]:
# Use Print to look at individual events
printer = Print(verbose=False)

print("First event in detail:")
printer.print_n_events(data, n_events=1)

**Understanding the output:**
- Each line shows a branch and its values for the event
- `[value]` means a single-element array (this event has 1 track)
- `[[value]]` means nested arrays (event → track → segment)
- Field names follow ROOT structure (e.g., `mom.fCoordinates.fX`)

## Step 3: Extract Basic Quantities

Let's extract some physics quantities we'll use later.

In [ ]:
# Track momentum magnitude
track_mom = data.trk.mom.mag
print(f"Track momentum shape: {track_mom.type}")
print(f"First 5 events: {track_mom[:5]}")
print()

# Track quality
track_qual = data.trkqual.result
print(f"Track quality shape: {track_qual.type}")
print(f"First 5 events: {track_qual[:5]}")
print()

# Number of active planes
n_active = data.trk.nactive
print(f"N active planes shape: {n_active.type}")
print(f"First 5 events: {n_active[:5]}")

**Key concept:** These are all *jagged arrays* - each event can have a different number of tracks. We'll need to handle this when making selections and plots.

## Step 4: Apply Physics Cuts

Now let's apply some basic physics selections using `pyselect`.

In [ ]:
# Create selector
selector = Select()

# Define particle type selections
is_electron = selector.is_electron(data)
is_positron = selector.is_positron(data)
is_mu_minus = selector.is_mu_minus(data)

# Geometry selections
is_downstream = selector.is_downstream(data, branch_name='trksegs')

# Print statistics
print("Selection statistics:")
print(f"Total events: {len(data)}")
print(f"  Electrons (e-): {ak.sum(is_electron)} ({100*ak.sum(is_electron)/len(data):.1f}%)")
print(f"  Positrons (e+): {ak.sum(is_positron)} ({100*ak.sum(is_positron)/len(data):.1f}%)")
print(f"  Muons (μ-): {ak.sum(is_mu_minus)} ({100*ak.sum(is_mu_minus)/len(data):.1f}%)")
print(f"  Downstream: {ak.sum(is_downstream)} ({100*ak.sum(is_downstream)/len(data):.1f}%)")

**What are these masks?**
- Each selection returns a boolean array (True/False for each event)
- These can be combined with `&` (AND) and `|` (OR)
- Apply them to `data` to filter events

In [ ]:
# Example: apply a mask
electron_data = data[is_electron]
print(f"Selected {len(electron_data)} electron events out of {len(data)} total")

# Can combine masks
electron_downstream = data[is_electron & is_downstream]
print(f"Electron AND downstream: {len(electron_downstream)} events")

## Step 5: Build a Cut Flow

For more complex analyses, use `CutManager` to track all your cuts and their efficiencies.

In [ ]:
# Create cut manager
cuts = CutManager(verbosity=1)

# Add cuts in sequence
# Note: These are event-level masks

# Cut 1: Has at least one track
has_track = ak.num(data.trk) > 0
cuts.add_cut(
    name="has_track",
    description="Event has at least one track",
    mask=has_track,
    active=True,
    group="basic"
)

# Cut 2: Electron candidate
cuts.add_cut(
    name="electron",
    description="Electron candidate (PDG code)",
    mask=is_electron,
    active=True,
    group="pid"
)

# Cut 3: Downstream track
cuts.add_cut(
    name="downstream",
    description="Downstream track segment",
    mask=is_downstream,
    active=True,
    group="geometry"
)

# Cut 4: Track quality
good_quality = ak.any(data.trkqual.result > QUALITY_CUT, axis=-1)
cuts.add_cut(
    name="quality",
    description=f"Track quality > {QUALITY_CUT}",
    mask=good_quality,
    active=True,
    group="quality"
)

# Cut 5: Momentum window
good_momentum = ak.any(
    (data.trk.mom.mag > MOMENTUM_MIN) & (data.trk.mom.mag < MOMENTUM_MAX),
    axis=-1
)
cuts.add_cut(
    name="momentum",
    description=f"{MOMENTUM_MIN} < p < {MOMENTUM_MAX} MeV/c",
    mask=good_momentum,
    active=True,
    group="kinematics"
)

print("\n✅ All cuts defined")

In [ ]:
# Generate and display the cut flow
cut_flow = cuts.create_cut_flow(data)
df_cut_flow = cuts.format_cut_flow(cut_flow)

print("\nCut Flow:")
print(df_cut_flow)

**Understanding the cut flow:**
- **N Events**: Starting number (before this cut)
- **N Pass**: How many events pass this cut
- **N Fail**: How many events fail this cut
- **Efficiency**: Pass / Events (for this cut only)
- **Cumulative Eff**: Total efficiency from all cuts so far

This table is crucial for understanding where you lose events!

In [ ]:
# Get the final selection
final_mask = cuts.combine_cuts()
selected_data = data[final_mask]

print(f"\nAfter all cuts: {len(selected_data)} events ({100*len(selected_data)/len(data):.2f}%)")

### Exercise: Try modifying cuts

Try changing the cut values and see how the efficiency changes:
- Change `QUALITY_CUT` to 0.7 or 0.9
- Change the momentum window
- Toggle cuts on/off with `cuts.toggle_cut({"quality": False})`

## Step 6: Calculate Derived Quantities

Let's calculate some physics quantities using our selected events.

In [ ]:
# Initialize vector calculator
vec = Vector()

# Get momentum and position vectors
mom_vec = vec.get_vector(selected_data.trksegs, 'mom')
pos_vec = vec.get_vector(selected_data.trksegs, 'pos')

print("Vector shapes:")
print(f"Momentum: {mom_vec.type}")
print(f"Position: {pos_vec.type}")

In [ ]:
# Calculate transverse momentum
mom_x = selected_data.trksegs.mom.fCoordinates.fX
mom_y = selected_data.trksegs.mom.fCoordinates.fY
mom_t = np.sqrt(mom_x**2 + mom_y**2)

print(f"Calculated p_T for {ak.sum(ak.num(mom_t))} track segments")

# Calculate radial position
pos_x = selected_data.trksegs.pos.fCoordinates.fX
pos_y = selected_data.trksegs.pos.fCoordinates.fY
pos_r = np.sqrt(pos_x**2 + pos_y**2)

print(f"Calculated r for {ak.sum(ak.num(pos_r))} track segments")

## Step 7: Create Plots

Time to visualize our results!

In [ ]:
# Create plotter
plotter = Plot()

### Plot 1: Track Momentum Distribution

In [ ]:
# Flatten the jagged array (events * tracks → all tracks)
mom_flat = ak.flatten(selected_data.trk.mom.mag)

print(f"Plotting {len(mom_flat)} tracks")

plotter.plot_1D(
    array=mom_flat,
    nbins=100,
    xmin=MOMENTUM_MIN - 5,
    xmax=MOMENTUM_MAX + 5,
    xlabel="Track momentum [MeV/c]",
    ylabel=f"Tracks / {(MOMENTUM_MAX - MOMENTUM_MIN + 10)/100:.2f} MeV/c",
    title="Selected Electron Momentum",
    col='steelblue',
    stat_box=True,
    show=True
)

### Plot 2: Track Quality Distribution

In [ ]:
qual_flat = ak.flatten(selected_data.trkqual.result)

plotter.plot_1D(
    array=qual_flat,
    nbins=50,
    xmin=0,
    xmax=1,
    xlabel="Track Quality MVA",
    ylabel="Tracks / 0.02",
    title="Track Quality Distribution",
    col='forestgreen',
    stat_box=True,
    show=True
)

### Plot 3: Compare Particle Types

In [ ]:
# Get momentum for different particle types
electron_mom = ak.flatten(data[is_electron].trk.mom.mag)
positron_mom = ak.flatten(data[is_positron].trk.mom.mag)

# Create overlay
hists_dict = {
    "e⁻": electron_mom,
    "e⁺": positron_mom
}

plotter.plot_1D_overlay(
    hists_dict=hists_dict,
    nbins=100,
    xmin=95,
    xmax=110,
    xlabel="Track momentum [MeV/c]",
    ylabel="Normalized counts",
    title="Momentum: Electrons vs Positrons",
    norm_by_area=True,
    leg_pos='upper left',
    show=True
)

### Plot 4: 2D Momentum vs Quality

In [ ]:
mom_2d = ak.flatten(selected_data.trk.mom.mag)
qual_2d = ak.flatten(selected_data.trkqual.result)

plotter.plot_2D(
    x=mom_2d,
    y=qual_2d,
    nbins_x=50,
    xmin=MOMENTUM_MIN - 5,
    xmax=MOMENTUM_MAX + 5,
    nbins_y=50,
    ymin=0,
    ymax=1,
    xlabel="Track momentum [MeV/c]",
    ylabel="Track Quality MVA",
    title="Momentum vs Quality",
    cmap='viridis',
    colorbar=True,
    show=True
)

### Plot 5: Multi-panel Summary

In [ ]:
# Create a 2x2 summary plot
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel 1: Momentum
plotter.plot_1D(
    array=ak.flatten(selected_data.trk.mom.mag),
    nbins=50,
    xmin=MOMENTUM_MIN - 5,
    xmax=MOMENTUM_MAX + 5,
    xlabel="Momentum [MeV/c]",
    title="Track Momentum",
    col='steelblue',
    ax=axes[0, 0],
    show=False
)

# Panel 2: Track quality
plotter.plot_1D(
    array=ak.flatten(selected_data.trkqual.result),
    nbins=50,
    xmin=0,
    xmax=1,
    xlabel="Track Quality MVA",
    title="Track Quality",
    col='forestgreen',
    ax=axes[0, 1],
    show=False
)

# Panel 3: Number of active planes
plotter.plot_1D(
    array=ak.flatten(selected_data.trk.nactive),
    nbins=50,
    xmin=0,
    xmax=100,
    xlabel="N Active Planes",
    title="Hit Count",
    col='coral',
    ax=axes[1, 0],
    show=False
)

# Panel 4: Transverse momentum
plotter.plot_1D(
    array=ak.flatten(mom_t),
    nbins=50,
    xmin=0,
    xmax=MOMENTUM_MAX + 5,
    xlabel="Transverse momentum [MeV/c]",
    title="Transverse Momentum",
    col='purple',
    ax=axes[1, 1],
    show=False
)

plt.tight_layout()
plt.savefig("analysis_summary.png", dpi=300)
plt.show()

print("\n✅ Saved: analysis_summary.png")

## Step 8: Save Results

Let's save our cut flow and some summary statistics.

In [ ]:
# Save cut flow to CSV
df_cut_flow.to_csv("cut_flow.csv", index=False)
print("✅ Saved: cut_flow.csv")

# Create and save summary statistics
import pandas as pd

summary = pd.DataFrame({
    'Quantity': [
        'Total events',
        'Selected events',
        'Overall efficiency',
        'Mean momentum [MeV/c]',
        'Mean track quality',
        'Mean N active planes'
    ],
    'Value': [
        len(data),
        len(selected_data),
        f"{100*len(selected_data)/len(data):.2f}%",
        f"{ak.mean(ak.flatten(selected_data.trk.mom.mag)):.2f}",
        f"{ak.mean(ak.flatten(selected_data.trkqual.result)):.3f}",
        f"{ak.mean(ak.flatten(selected_data.trk.nactive)):.1f}"
    ]
})

summary.to_csv("analysis_summary.csv", index=False)
print("✅ Saved: analysis_summary.csv")
print("\nSummary:")
print(summary)

## Summary

Congratulations! You've completed your first Mu2e analysis. You learned how to:

✅ Load EventNtuple data with `pyprocess.Processor`  
✅ Inspect data structure with `pyprint.Print`  
✅ Apply physics cuts with `pyselect.Select`  
✅ Track cut flow with `pycut.CutManager`  
✅ Calculate derived quantities  
✅ Create publication-quality plots with `pyplot.Plot`  
✅ Export results to files  

## Next Steps

Try modifying this notebook:
- Change the cut values and see how results change
- Add new cuts (e.g., on `nactive`, other quality metrics)
- Compare different particle types
- Add more derived quantities
- Try different plot styles and configurations

**Resources:**
- [Understanding the Stack](01-UnderstandingTheStack.md)
- [Next Steps Guide](03-NextSteps.md)
- [pyutils README](../../pyutils/README.md)
- [pyutils Examples](../../pyutils/examples/notebooks/)
- Slack: #analysis-tools

Happy analyzing!

---

## Exercises

Try these on your own:

### Exercise 1: Momentum resolution
If you have MC data with truth information, calculate and plot:
- Momentum resolution: (p_reco - p_true) / p_true
- As a function of momentum

### Exercise 2: Cut optimization
Create a scan over quality cut values (0.5 to 0.95) and:
- Plot efficiency vs quality cut
- Find the optimal value for your analysis

### Exercise 3: Data vs MC comparison
If you have both data and MC files:
- Load both
- Apply the same cuts
- Create overlay plots comparing distributions

### Exercise 4: Spatial distributions
Using track segment positions:
- Plot the x-y position at tracker entrance
- Plot the radial position distribution
- Plot z vs r

Good luck!